# 0.准备工作

In [61]:
from typing import Collection

from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =====================1.准备文档===========================
    with open("./resources/评估.md", "r", encoding="utf-8") as f:
        doc = f.read()

    # =====================2.切分文档===========================
    # 切分依据，这里是按照markdown的三级标题进行切分
    headers_to_split_on = [
        ("#", "H1"),
        ("##", "H2"),
        ("###", "H3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False  # 是否在正文中去除标题，false不去除
    )

    # 切分文档
    docs = markdown_splitter.split_text(doc)

    return docs


docs = prepare_docs()

In [62]:
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(model="qwen3-embedding:4b", dimensions=1024, )

print(docs[0].model_dump_json(indent=2))


{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（一）教育学萌芽时期代表作"
  },
  "page_content": "# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。",
  "type": "Document"
}


### 到上面这里为止，所有的准备工作都完成了,下一步就是使用milvus sdk去操作milvus了

# 1.初始化client

In [63]:
from pymilvus import MilvusClient

# 初始化client客户端
client = MilvusClient(uri="http://127.0.0.1:19530")  ## 地址是你自己的部署地址

print(client.get_server_version())

# Collection名称
collection_name = "my_collection_1"

3.0-20261030-af15c80751


# 2.创建Collection

In [64]:
# 利用刚才创建好的milvus client客户端来创建collection
from pymilvus import Function, FunctionType, DataType


# 下面这段代码不用死记硬背，到时候ctrl + c + v 然后去改就行了
def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加BM25函数，可以自动把原始文档转稀疏向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)
    # 1.7.添加Embedding函数，可以自动把原始文档转稠密向量存储
    text_embedding_function = Function(
        name="ali",  # Embedding函数的名称
        function_type=FunctionType.TEXTEMBEDDING,  # 函数类型，选择embedding function
        input_field_names=["content"],  # 用来转稠密向量的原始文本字段的名称
        output_field_names=["dense"],  # 存储稠密向量字段的名称
        params={  # 向量模型提供者的相关参数
            "provider": "dashscope",  # 模型提供者名称，百炼的话必须是"dashscope"
            "model_name": "text-embedding-v3",  # 向量模型名称
            "dim": "1024",  # 其它可选字段: 例如向量维度
        }
    )
    # 将配置好的Embedding函数添加到现有的Schema中
    schema.add_function(text_embedding_function)

    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )


create_collection()

# 3.删除和查看Collection

In [10]:
client.list_collections()

['my_collection_1']

In [11]:
# client.drop_collection(collection_name)

# 4.新增数据

In [13]:
# 1.把Document的page_content生成稠密向量（可选操作,如果用了Embedding Function，就不用自己转向量）
# dense_vectors = embeddings.embed_documents([doc.page_content for doc in docs])

In [65]:
# 2.把Document转成Dict，符合Milvus的schema约束
# for doc in docs:
#     print(doc.model_dump_json(indent=2))

data = [
    {
        "id": i,
        "h2": doc.metadata.get("H2"),
        "content": doc.page_content,
        # 这里不需要来加入稠密向量和稀疏向量，因为milvus会自动帮我们把content转成dense稠密向量和和sparse稀疏向量
    }
    for i, doc in enumerate(docs, start=1)
]

In [66]:
# 直接调用insert方法来添加数据
client.insert(collection_name, data)

{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'cost': 0}

# 5.upsert和delete

upsert会根据id删除旧数据，再添加新数据。

In [66]:
# 1.准备两条数据
# id为1 的数据，修改其H2字段，模拟数据更新
data1=data[0]
data1["h2"]="测试一下"

# id为2的数据，将其id改为88，模拟新增数据
data2=data[1]
data2["id"]=88
data2["h2"]="再测试一下"

In [67]:
# 2.调用upsert方法
client.upsert(collection_name, [data1,data2])

{'upsert_count': 2, 'cost': 0, 'ids': [1, 88]}

In [68]:
# 3.尝试delete
client.delete(collection_name, ids=[88,])

{'delete_count': 1, 'cost': 0}

# 6.稠密检索

In [53]:
# 稠密向量检索
def dense_search(query:str):
    # 1.把问题query转成向量（如果在创建collection的那一步已经添加过embedding函数，就是会自动把content转成dense和sparse的那个函数），如果需要用自定义的向量化模型来处理，这一步就需要来做手动向量化
    query_vector=embeddings.embed_documents([query])

    # 2.检索文档
    result=client.search(
        collection_name=collection_name,
        data=[query],       # 这里data表示的是查询条件
        anns_field="dense", # 这里anns_field表示的是查询条件对应的向量字段，填dense就是稠密检索，填sparse就是稀疏检索
        limit=3,            # 这里limit表示的是返回的文档数量
        output_fields=["h2","content"], # 这里output_fields表示的是返回的文档字段
    )

    # 3.打印
    # 这里用两层for循环来遍历结果是因为search的返回值result是一个List[List[dict]]类型的数据
    for hits in result:         # 这里hit在英文中表示的是命中的意思，在搜索、检索、数据库领域，一条匹配的结果就叫做一个hit
        for hit in hits:
            print(hit)


In [54]:
dense_search("教育的负面作用是什么")

{'id': 15, 'distance': 0.6704650521278381, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}
{'id': 21, 'distance': 0.6083123087882996, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 16, 'distance': 0.5939378142356873, 'entity': {'content': '## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。', 'h2': '第二节 教育和社会的关系'}}


In [55]:
dense_search("给教师的建议是谁写的")      # 这里给教师的建议是一本书的名字，从结果来看，稠密检索对这种类型的检索效果就不是很好了

{'id': 1, 'distance': 0.5570199489593506, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。'}}
{'id': 6, 'distance': 0.5557209253311157, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （六）希腊三贤  \n**苏格拉底**：产婆术（谈话法），分三步：讽刺—定义—助产术。国外启发式教育第一人。  \n**柏拉图**：《理想国》提出普及教育的主张，教育目的是培养哲学王。  \n**亚里士多德**：提出自由教育，提倡对学生进行和谐全面发展的教育，灵魂说。'}}
{'id': 8, 'distance': 0.5421857833862305, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教

# 7.稀疏检索

In [56]:
# 稀疏向量检索

# 只改动稠密检索代码中的search函数的一个参数就能得到稀疏检索
def sparse_search(query:str):
    result = client.search(
        collection_name=collection_name,
        data=[query],
        anns_field="sparse", # 这里anns_field表示的是查询条件对应的向量字段，填dense就是稠密检索，填sparse就是稀疏检索
        limit=3,            # 这里limit表示的是返回的文档数量
        output_fields=["h2","content"], # 这里output_fields表示的是返回的文档字段
    )

    for hits in result:
        for hit in hits:
            print(hit)


In [57]:
sparse_search("教育的负面作用是什么")

{'id': 21, 'distance': 1.811586856842041, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。'}}
{'id': 20, 'distance': 1.777803659439087, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （五）人口与教育的关系  \n人口对教育的作用主要表现在：人口的数量和增长是决定教育事业的规模、速度的一个重要因素；人口的质量影响教育的质量；人口的结构影响着教育的发展；人口流动影响着教育的发展。  \n教育对人口的作用主要表现在：控制人口的数量；提高人口的质量；改善人口结构；调整人才构成与流动等作用。'}}
{'id': 5, 'distance': 1.7445214986801147, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。'}}


In [58]:
sparse_search("给教师的建议是谁写的")      # 这里给教师的建议是一本书的名字，从结果来看，稀疏检索对这种类型的检索效果就很好了

{'id': 8, 'distance': 4.901364803314209, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。'}}
{'id': 7, 'distance': 2.466482400894165, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （七）教育学创立时期代表人物及主要思想  \n**昆体良**：西方第一个专门论述教育问题的教育家。  \n**培根**：在《论科学的价值和发展》中首次把“教育学”作为一门独立的科学确立下来。  \n**夸美纽斯**（教育学之父、教育史上哥白尼）：代表作《大教学论》是教育学

# 8.混合检索

## 8.1.通用混合检索函数(函数会接收一个融合策略的参数，将来只需要修改这个参数就可以使用不同的融合重排策略）

In [70]:
from pymilvus import AnnSearchRequest

def hybrid_search(query:str,ranker):        # ranker表示融合策略的参数
    # 1.创建稠密、稀疏请求
    query_dense_vector = embeddings.embed_query(query)

    # 稠密请求
    dense_request = AnnSearchRequest(       # 这里不是client.search()函数，而是AnnSearchRequest()函数
        data=[query],
        anns_field="dense",
        limit=3,
        param={},
    )
    # 稀疏请求
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={},
        limit=3,
    )

    # 混合在一起
    reqs=[dense_request,sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name="my_collection_1",
        reqs=reqs,          # 混合检索的请求
        ranker=ranker,      # 混合重排策略
        limit=2,
        output_fields=["content","h2"]
    )

    for hits in res:
        print("TopK results：")
        for hit in hits:
            print(hit)

## 8.1.加权


In [71]:
# 加权求和的策略milvus已经帮我们实现好了，导入相关包就可以了
from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)

# 测试
hybrid_search("教育的负面作用是什么？",ranker)


TopK results：
{'id': 21, 'distance': 0.7375719547271729, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。'}}
{'id': 15, 'distance': 0.41711121797561646, 'entity': {'h2': '第一节 教育的功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'}}


In [72]:
hybrid_search("给教师的建议是谁写的",ranker)

TopK results：
{'id': 8, 'distance': 0.43787047266960144, 'entity': {'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。', 'h2': '第一节 中外教育家及其教育思想'}}
{'id': 1, 'distance': 0.38925498723983765, 'entity': {'content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。', 'h2': '第一节 中外教育家及其教育思想'}}


## 8.2.RRF融合

In [73]:
from pymilvus import RRFRanker

# 定义RRF融合策略
rrf_ranker = RRFRanker()        # 里面的常量值k=60，可以对这个default值进行调整来调整召回效果

# 实现RRF检索，传入参数就好了
hybrid_search("教育的负面作用是什么？",rrf_ranker)

TopK results：
{'id': 21, 'distance': 0.032522473484277725, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 15, 'distance': 0.016393441706895828, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}


In [74]:
hybrid_search("给教师的建议是谁写的",rrf_ranker)

TopK results：
{'id': 1, 'distance': 0.016393441706895828, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。'}}
{'id': 8, 'distance': 0.016393441706895828, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。'}}


## 8.3.Cross-Encoder融合重排

In [75]:
from pymilvus import Function,FunctionType

def creat_cross_encoder_ranker(query:str):
    return Function(
        name="dashscope_semantic_ranker",       # ranker名字，自定义就行
        input_field_names=["content"],          # 原始文档字段
        function_type=FunctionType.RERANK,      # ranker类型,这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称，可以在阿里云百炼的模型广场中查找
            "queries": [query],  # 查询条件
            "max_client_batch_size": 5,  # 单批次中处理的文档最大数量
        },
    )

In [76]:
# 用户问题
query="教育的负面作用是什么？"
# 创建ranker
ranker = creat_cross_encoder_ranker(query)
# 混合检索
hybrid_search(query,ranker)

TopK results：
{'id': 15, 'distance': 0.290463387966156, 'entity': {'h2': '第一节 教育的功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'}}
{'id': 20, 'distance': 0.1413116455078125, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （五）人口与教育的关系  \n人口对教育的作用主要表现在：人口的数量和增长是决定教育事业的规模、速度的一个重要因素；人口的质量影响教育的质量；人口的结构影响着教育的发展；人口流动影响着教育的发展。  \n教育对人口的作用主要表现在：控制人口的数量；提高人口的质量；改善人口结构；调整人才构成与流动等作用。'}}
